# 04 · Embeddings, prototypes, k-NN and autoencoders (drill)

**Style B: blind-code drill.** Write each piece from memory, run the check, delete your code.
Reference: `../solutions/drills/04_embeddings_prototypes.py`.

**Traces to** `notes/ioai-task-patterns.md` P4, P6 · T24-O-BOBAI (frozen head + cached embeddings + "averages
and distances"), T24-O-HYPER (unsupervised autoencoder features). Data embeddings are a syllabus "Both" item,
so sections 2–3 are from scratch in torch.

Total suggested time: **50 min**.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
cifar = load_dataset("uoft-cs/cifar10")
def balanced(split, per_class, seed):
    labels = np.array(split["label"]); rng = np.random.default_rng(seed)
    return split.select(np.concatenate([rng.choice(np.flatnonzero(labels == c), per_class, replace=False) for c in range(10)]))
train, test = balanced(cifar["train"], 150, 0), balanced(cifar["test"], 60, 1)
y_tr, y_te = torch.tensor(train["label"]), torch.tensor(test["label"])

## 1. Frozen ResNet18 embedder ⏱ 10 min

`embed_images(pil_images) -> Tensor (n, 512)` on the CPU. Use torchvision's ResNet18 with its default ImageNet
weights and **their** preprocessing transform, drop the classification layer, and batch the forward passes.
No gradients.

In [ ]:
E_tr, E_te = embed_images(train["img"]), embed_images(test["img"])
assert E_tr.shape == (1500, 512) and E_tr.device.type == "cpu" and not E_tr.requires_grad
# a different batch size changes GPU conv algorithms (~1e-3); train-mode BatchNorm would change far more
assert torch.allclose(embed_images(train["img"][:5]), E_tr[:5], rtol=1e-2, atol=1e-2), "embedding is not deterministic (eval mode?)"
acc = KNeighborsClassifier(10).fit(E_tr, y_tr).score(E_te, y_te)
assert acc > 0.6, acc
print(f"section 1 ok (10-NN accuracy {acc:.3f})")

## 2. Prototype classifier ⏱ 6 min

- `fit_prototypes(E, y, n_classes) -> P (n_classes, d)`: mean **L2-normalised** embedding per class.
- `predict_prototypes(E, P) -> labels`: the class of the most cosine-similar prototype.

In [ ]:
P = fit_prototypes(E_tr, y_tr, 10)
Zc = E_tr / E_tr.norm(dim=1, keepdim=True)
assert P.shape == (10, 512) and torch.allclose(P[4], Zc[y_tr == 4].mean(0), atol=1e-5)
pred = predict_prototypes(E_te, P)
sims = (E_te / E_te.norm(dim=1, keepdim=True)) @ (P / P.norm(dim=1, keepdim=True)).T
assert torch.equal(pred, sims.argmax(1))
print(f"section 2 ok (accuracy {(pred == y_te).float().mean():.3f})")

## 3. k-NN from scratch ⏱ 8 min

`knn_predict(E_train, y_train, E_test, k) -> labels` with **cosine** similarity and a majority vote (break ties
toward the smallest class id). Torch only. It must agree with sklearn's `KNeighborsClassifier(k, metric="cosine")`
on ≥ 97 % of points.

In [ ]:
mine = knn_predict(E_tr, y_tr, E_te, 7)
ref = torch.tensor(KNeighborsClassifier(7, metric="cosine").fit(E_tr, y_tr).predict(E_te))
agree = (mine == ref).float().mean().item()
assert mine.shape == y_te.shape and agree >= 0.97, agree
print(f"section 3 ok (agreement {agree:.3f})")

## 4. Autoencoder features ⏱ 12 min

`train_autoencoder(X, code_dim=32, steps=1500) -> encoder`: train an MLP autoencoder on the rows of `X` with an MSE
reconstruction loss, and return the **encoder** (`nn.Module`, `(n, 512) -> (n, code_dim)`). The check needs 10-NN
accuracy on the codes to reach ≥ 90 % of 10-NN accuracy on the raw embeddings.

In [ ]:
Zall = F.normalize(torch.cat([E_tr, E_te]), dim=1)
enc = train_autoencoder(Zall, 32)
with torch.no_grad():
    C_tr, C_te = enc(F.normalize(E_tr, dim=1)), enc(F.normalize(E_te, dim=1))
assert C_tr.shape == (1500, 32)
raw = KNeighborsClassifier(10).fit(F.normalize(E_tr, dim=1), y_tr).score(F.normalize(E_te, dim=1), y_te)
code = KNeighborsClassifier(10).fit(C_tr, y_tr).score(C_te, y_te)
assert code >= 0.9 * raw, (code, raw)
print(f"section 4 ok (codes {code:.3f} vs raw {raw:.3f})")

## 5. Seven classes from a frozen five-way head ⏱ 12 min

`head` below is a frozen `Linear(512, 5)` for classes 0–4. Classes 5 and 6 come with `E_few, y_few` (15 examples
each). Write `seven_way(E) -> labels in 0..6` that uses the head plus means and distances only (no new trained
parameters). The check needs 7-way macro F1 ≥ 0.65 on the test images of classes 0–6.

In [ ]:
old = y_tr < 5
head = nn.Linear(512, 5); _o = torch.optim.Adam(head.parameters(), 1e-2)
for _ in range(300):
    l = F.cross_entropy(head(F.normalize(E_tr[old], dim=1)), y_tr[old]); _o.zero_grad(); l.backward(); _o.step()
head.requires_grad_(False)
few_idx = torch.cat([torch.nonzero(y_tr == c).flatten()[:15] for c in (5, 6)])
E_few, y_few = E_tr[few_idx], y_tr[few_idx]
E_old, y_old = E_tr[old], y_tr[old]
# the head expects L2-normalised embeddings: head(F.normalize(E, dim=1))

In [ ]:
m7 = y_te < 7
pred = torch.as_tensor(seven_way(E_te[m7]))
f1 = f1_score(y_te[m7], pred, average="macro")
assert f1 >= 0.65, f1
print(f"section 5 ok (7-way macro F1 {f1:.3f})")